In [4]:
# codigo inicial que unificou as bases de vendas e vistoria, criou as colunas de ágio absoluto e percentual, 
# e removeu os registros duplicados dos itens agrupados
# removeu os imoveis da tipologia apartamento/nao é caracteristica da empresa. Venda por convenio de outra instituicao
# teste usando o codigo da destinacao original, via onehot encoding

import pandas as pd

tabela = pd.read_csv("base_vendas_atividade2_final.csv", sep=";", encoding="latin-1")
vistoria = pd.read_csv("base_vistorias_atividade2_final.csv", sep=";", encoding="latin-1")

tabela = tabela.merge(
    vistoria[["CD_IMOVEL_URBANO", "SIM_SITIMO_DS"]],
    how="left",
    left_on="CD_IMOVEL",
    right_on="CD_IMOVEL_URBANO"
)

# # remove a coluna duplicada da chave
tabela = tabela.drop(columns=["CD_IMOVEL_URBANO", "CD_IMOVEL"])
tabela = tabela.rename(columns={"SIM_SITIMO_DS": "SITUACAO_VISTORIA"})
# tabela["SITUACAO_VISTORIA"] = tabela["SITUACAO_VISTORIA"].fillna("SEM_VISTORIA")


# # colunas que definem a duplicidade: mesmo ano, mesmo edital e mesmo item do edital (item de edital com imoveis agrupados)
cols = ["ANO_VENDA", "NR_EDITAL", "ITEM_EDITAL"]
# # máscara: True para linhas que aparecem em duplicidade (em qualquer posição do grupo)
mask_dup = tabela.duplicated(subset=cols, keep=False)
# # remove TODAS as linhas duplicadas desses grupos
tabela_sem_dups = tabela.loc[~mask_dup].copy()

# # Foram removidos 145 itens que constavam como items agrupados, de um total de 2982, restando 2837 registros 
# print("Linhas originais:", len(tabela))
# print("Linhas removidas:", mask_dup.sum())
# print("Linhas finais:", len(tabela_sem_dups))

tabela = tabela_sem_dups
# tabela["NR_EDITAL"] = tabela["NR_EDITAL"].astype(str) + "/" + tabela["ANO_VENDA"].astype(str)

# #criacao da coluna AGIO ABSOLUTO , esta coluna deve ser retirada do treino
tabela["AGIO_ABSOLUTO"] = tabela["VALOR_VENDA"] - tabela["VALOR_LAUDO"]
tabela["AGIO_PERCENTUAL"] = ((tabela["VALOR_VENDA"] - tabela["VALOR_LAUDO"]) / tabela["VALOR_LAUDO"]) * 100

#retirando os apartamentos da base (14 registros) que tem área máxima de construção igual a zero e área base igual a zero, ou seja, não tem área construída, o que é um erro de cadastro
# tabela[~(tabela["AREA_MAX_CONSTR"] == 0) & (tabela["AREA_BASE"] == 0)]
tabela = tabela[~((tabela["AREA_MAX_CONSTR"] == 0) & (tabela["AREA_BASE"] == 0))]   

# tabela["COD_DESTINACAO_IMOVEL"].nunique()
# tabela["DS_CIDADE"].nunique()
# tabela["DS_SETOR"].nunique()
tabela["SITUACAO_VISTORIA"].nunique()


# tabela.to_csv('dados.csv', sep=";", index=False, encoding='utf-8')





7

In [5]:
# Analisar correlação entre as variáveis numéricas e a variável target "VALOR_VENDA"

colunas = [
    "VALOR_LAUDO",
    "AREA_MAX_CONSTR",
    "AREA_BASE",
    "AREA",
    "PERCENTUAL_ENTRADA",
    "QTD_OFERTAS",
    "ANO_VENDA",
    "NR_EDITAL",
    "ITEM_EDITAL",
    "AGIO_ABSOLUTO",
    "AGIO_PERCENTUAL",
    "VALOR_VENDA"   # target Y
]

# cria um dataframe apenas com essas colunas
df_corr = tabela[colunas].copy()

# calcula a matriz de correlação
matriz_corr = df_corr.corr(method="pearson")
# correlação de todas as variáveis com a target
corr_com_y = matriz_corr["VALOR_VENDA"].sort_values(ascending=False)

display(matriz_corr)
corr_com_y_df = corr_com_y.reset_index()
corr_com_y_df.columns = ["Variavel", "Correlacao_com_VALOR_VENDA"]

display(corr_com_y_df)

,VALOR_LAUDO,AREA_MAX_CONSTR,AREA_BASE,AREA,PERCENTUAL_ENTRADA,QTD_OFERTAS,ANO_VENDA,NR_EDITAL,ITEM_EDITAL,AGIO_ABSOLUTO,AGIO_PERCENTUAL,VALOR_VENDA
VALOR_LAUDO,1.000000,0.072238,0.110862,0.019533,0.309247,-0.012628,-0.056298,0.026902,-0.073819,0.794526,-0.007029,0.993811
AREA_MAX_CONSTR,0.072238,1.000000,0.545245,0.636855,0.063999,-0.051631,-0.036646,-0.034761,0.013767,0.035058,-0.045100,0.067704
AREA_BASE,0.110862,0.545245,1.000000,0.388395,0.125259,-0.029177,-0.070341,-0.033809,-0.035726,0.088615,-0.031332,0.110273
AREA,0.019533,0.636855,0.388395,1.000000,-0.013671,-0.038916,-0.034911,-0.012632,0.029471,0.003979,-0.030858,0.017301
PERCENTUAL_ENTRADA,0.309247,0.063999,0.125259,-0.013671,1.000000,0.027813,-0.059106,0.020594,-0.148056,0.493256,0.029864,0.352620
QTD_OFERTAS,-0.012628,-0.051631,-0.029177,-0.038916,0.027813,1.000000,-0.061342,-0.023576,-0.205834,0.085027,0.458085,0.004841
ANO_VENDA,-0.056298,-0.036646,-0.070341,-0.034911,-0.059106,-0.061342,1.000000,0.062716,0.002705,-0.065219,0.014296,-0.059698
NR_EDITAL,0.026902,-0.034761,-0.033809,-0.012632,0.020594,-0.023576,0.062716,1.000000,0.030386,0.033077,-0.002085,0.028876
ITEM_EDITAL,-0.073819,0.013767,-0.035726,0.029471,-0.148056,-0.205834,0.002705,0.030386,1.000000,-0.144233,-0.175121,-0.089019
AGIO_ABSOLUTO,0.794526,0.035058,0.088615,0.003979,0.493256,0.085027,-0.065219,0.033077,-0.144233,1.000000,0.200630,0.857065


,Variavel,Correlacao_com_VALOR_VENDA
0,VALOR_VENDA,1.000000
1,VALOR_LAUDO,0.993811
2,AGIO_ABSOLUTO,0.857065
3,PERCENTUAL_ENTRADA,0.352620
4,AREA_BASE,0.110273
5,AREA_MAX_CONSTR,0.067704
6,AGIO_PERCENTUAL,0.030740
7,NR_EDITAL,0.028876
8,AREA,0.017301
9,QTD_OFERTAS,0.004841


In [3]:
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score, mean_absolute_percentage_error
from sklearn.ensemble import RandomForestRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor  
import numpy as np

														
# identificar tipos de colunas
variaveis_numericas = ["AREA_MAX_CONSTR", "AREA_BASE", "AREA", "PERCENTUAL_ENTRADA"]
variaveis_categoricas = ["DS_CIDADE", "DS_SETOR", "COD_DESTINACAO_IMOVEL", "ANO_VENDA", "NR_EDITAL", "ITEM_EDITAL", "SITUACAO_VISTORIA"]

# y -> é a coluna da base de dados que eu quero prever
y = tabela["VALOR_VENDA"]
# x -> as colunas da base de dados que eu vou usar pra fazer a previsão
X = tabela.drop(columns=["VALOR_VENDA", "VALOR_LAUDO", "AGIO_ABSOLUTO", "AGIO_PERCENTUAL", "QTD_OFERTAS"])

X_treino, X_teste, y_treino, y_teste = train_test_split(
    X, y, test_size=0.2, random_state=42
)

escalonador = StandardScaler()
categorizador = OneHotEncoder(handle_unknown="ignore")
imputador_numerico = SimpleImputer(strategy="median")
imputador_categorico = SimpleImputer(strategy="most_frequent")

# preprocessamento
etapas_numericas = Pipeline(
    [
        ("imputer", imputador_numerico),
        ("scaler", escalonador),
    ]
)

etapas_categoricas = Pipeline(
    [
        ("imputer", imputador_categorico),
        ("encoder", categorizador),
    ]
)


preprocess = ColumnTransformer(
    transformers=[
        ("num", etapas_numericas, variaveis_numericas),
        ("cat", etapas_categoricas, variaveis_categoricas),
    ]
)

X_treino_transformado = preprocess.fit_transform(X_treino)
X_teste_transformado = preprocess.transform(X_teste)

In [ ]:
modelo = DecisionTreeRegressor(random_state=42)
param_grid = {
    "max_depth": [2, 3, 5, 7],
    "min_samples_split": [2, 5, 10, 15, 20]
}

grid_search = GridSearchCV(estimator=modelo, param_grid=param_grid, cv=5, scoring="neg_root_mean_squared_error")

grid_search.fit(X_treino_transformado, y_treino)
melhor_modelo = grid_search.best_estimator_

In [9]:
y_pred_treino = melhor_modelo.predict(X_treino_transformado)
rmse_treino = root_mean_squared_error(y_treino, y_pred_treino)
mape_treino = mean_absolute_percentage_error(y_treino, y_pred_treino)
mae_treino = mean_absolute_error(y_treino, y_pred_treino)
r2_treino = r2_score(y_treino, y_pred_treino)

print("Melhores hiperparâmetros:", grid_search.best_params_)
print("RMSE (Treino):", rmse_treino) 
print("MAPE (Treino):", mape_treino)
print("MAE (Treino):", mae_treino)
print("R² (Treino):", r2_treino)

Melhores hiperparâmetros: {'max_depth': 7, 'min_samples_split': 5}
RMSE (Treino): 612943.503247063
MAPE (Treino): 0.38098644343972676
MAE (Treino): 231024.69810584004
R² (Treino): 0.9954813887225226


In [10]:
y_pred_teste = melhor_modelo.predict(X_teste_transformado)

rmse = root_mean_squared_error(y_teste, y_pred_teste)
mape = mean_absolute_percentage_error(y_teste, y_pred_teste)
mae = mean_absolute_error(y_teste, y_pred_teste)
r2 = r2_score(y_teste, y_pred_teste)

print("MAPE Teste:", mape)
print("MAE Teste:", mae)
print("RMSE Teste:", rmse)
print("R² Teste:", r2)

MAPE Teste: 0.4926503939513165
MAE Teste: 528516.1002896513
RMSE Teste: 2026578.4024875162
R² Teste: 0.6874915632209444
